# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Method choice

I use **K-Means clustering** because my lane is Structured Content Archetype Clustering and there is no ground-truth archetype label in the warehouse. The goal is to group content pages with similar performance patterns across impressions, clicks, CTR, average search position, and position volatility.

The feature distributions are strongly skewed, especially impressions and clicks, so non-negative count features will be log-transformed before standardization. Standardization is then used so that features measured on different scales do not dominate K-Means distance calculations.

I will evaluate candidate cluster counts using silhouette score, cluster size, stability, and interpretability. Cluster names will be assigned only after inspecting the resulting cluster profiles; they are descriptive decision-support labels, not ground-truth classes.

The model is intended to discover performance archetypes that can support actions such as protect, improve, rewrite, merge, prune, or monitor. It does not establish causal relationships between page characteristics and performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )


HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection established.")
print("Using March 2026 development data.")

DuckDB connection established.
Using March 2026 development data.


In [2]:
# Build the same five-feature vector established during ML-05

feature_vector = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(
            NULLIF(gsc_avg_position, 0)
        ) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

clustering_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]

print("Rows:", len(feature_vector))
print("Clustering features:", clustering_features)

display(feature_vector.head())

print("\nMissing values:")
display(
    feature_vector[clustering_features]
    .isna()
    .sum()
    .to_frame("missing_rows")
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Clustering features: ['impressions', 'clicks', 'ctr', 'avg_position', 'position_volatility']


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_volatility
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.888929,2.119233
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.240351,1.376604
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,7.061594,3.686090
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.141844,6.155424,3.892530
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.343567,14.439705



Missing values:


,missing_rows
impressions,0
clicks,0
ctr,0
avg_position,1434
position_volatility,15181


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.